In [1]:
# =========================
# qcqp_generator_mixed.py  (LATEST VERSION)
#
# ✅ Option A: keep ORIGINAL l/u (do NOT overwrite)
# ✅ If bounds_exist=False: inject ONE PD (ellipsoid) CONSTRAINT (k>=1) and
#    compute/store artificial bounds ONCE in l_art/u_art (so solver won't recompute)
# ✅ Increase linear objectives via p_linear_objective: with prob p, set A0 = 0
# ✅ Reduce indefiniteness by making constraints mostly CONVEX (PSD) and some CONCAVE (NSD)
#    - For non-bilinear constraints (i>=1): choose curvature with probs
#        convex: p_convex_constr -> PSD
#        concave: p_concave_constr -> -PSD
#        otherwise -> original generation (possibly indefinite)
#    - For bilinear constraints: keep as-is (bilinear enforces indefiniteness)
#
# NOTE: Objective PSD/concave control is NOT added here (separate step).
# =========================

import numpy as np
import pandas as pd
from sklearn.datasets import make_spd_matrix
import networkx as nx
from networkx.algorithms import bipartite
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional, Tuple


# =============================================================================
# Graph / matrix helpers
# =============================================================================

def make_rng(seed: int):
    return np.random.default_rng(seed)


def adjacency_to_distance(A, rng, allow_zero_edge_weights: bool = False):
    if allow_zero_edge_weights:
        random_weights = rng.integers(-10, 11, size=A.shape)
    else:
        random_weights = rng.integers(1, 11, size=A.shape) * rng.choice([-1, 1], size=A.shape)

    D = np.where(A == 1, random_weights, 0)
    D = np.tril(D) + np.triu(D.T, 1)
    np.fill_diagonal(D, rng.integers(1, 11, size=D.shape[0]))
    return D, False


def generate_bipartite_distance_matrix(n, rng: np.random.Generator):
    k = rng.integers(2, n - 1)
    e = rng.integers(2, k * (n - k) + 1)
    seed_int = int(rng.integers(0, 2**31 - 1))
    B = bipartite.gnmk_random_graph(k, n - k, e, seed=seed_int)
    A = nx.to_numpy_array(B, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_tree_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    T = nx.random_tree(n, seed=seed_int)
    A = nx.to_numpy_array(T, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_planar_distance_matrix(n, rng: np.random.Generator):
    G = nx.complete_graph(n)
    edges = rng.permutation(list(G.edges)).tolist()
    is_planar = False
    while not is_planar and edges:
        edge = edges.pop()
        G.remove_edge(*edge)
        is_planar, _ = nx.check_planarity(G)
    if not nx.is_connected(G):
        remaining_edges = rng.permutation(list(nx.non_edges(G))).tolist()
        if remaining_edges:
            G.add_edge(*remaining_edges[0])
    A = nx.to_numpy_array(G, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_chordal_distance_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    T = nx.random_tree(n, seed=seed_int)
    for node in range(n):
        neighbors = list(nx.neighbors(T, node))
        if len(neighbors) >= 2:
            v, w = rng.choice(neighbors, size=2, replace=False)
            T.add_edge(v, w)
    A = nx.to_numpy_array(T, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_symmetric_matrix(n: int, rng: np.random.Generator):
    A = rng.uniform(-5, 5, size=(n, n))
    D = 0.5 * (A + A.T)
    return D, False


def generate_spd_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    D = make_spd_matrix(n, random_state=seed_int)
    return D, False


def _rand_Q(n, rng):
    Q, _ = np.linalg.qr(rng.uniform(0, 9, size=(n, n)))
    return Q


def diag_ordered_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    v[:n_neg] = -1
    return np.diag(v), False


def diag_random_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] = -1
    return np.diag(v), False


def diag_random_ints(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = rng.integers(1, 11, size=n).astype(float)
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] *= -1
    return np.diag(v), False


def eigvals_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    v[:n_neg] = -1
    L = np.diag(v)
    Q = _rand_Q(n, rng)
    return Q @ L @ Q.T, False


def eigvals_ints(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = rng.integers(1, 11, size=n).astype(float)
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] *= -1
    L = np.diag(v)
    Q = _rand_Q(n, rng)
    return Q @ L @ Q.T, False


# =============================================================================
# Artificial bounds (Cholesky-based)
# =============================================================================

def compute_artificial_bounds(A: np.ndarray, b: np.ndarray, c: float):
    """
    Bounds from ellipsoid:
        x^T A x + 2 b^T x + c <= 0
    assuming A is PD.
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1)
    n = A.shape[0]
    if A.shape != (n, n):
        raise ValueError("A must be square (n,n).")
    if b.size != n:
        raise ValueError("b has wrong dimension.")

    L = np.linalg.cholesky(A)

    def solve_A(rhs):
        rhs = np.asarray(rhs, dtype=float)
        y = np.linalg.solve(L, rhs)
        x = np.linalg.solve(L.T, y)
        return x

    Ainv_b = solve_A(b)
    b_Ainv_b = float(b @ Ainv_b)

    t = b_Ainv_b - float(c)
    if t < 0:
        raise ValueError("Invalid bounding: b^T A^-1 b - c must be non-negative.")

    sqrt_term = float(np.sqrt(max(t, 0.0)))

    Ainv = solve_A(np.eye(n))
    diag_Ainv = np.diag(Ainv)

    l = np.zeros(n, dtype=float)
    u = np.zeros(n, dtype=float)

    for i in range(n):
        di = float(diag_Ainv[i])
        if di < 0:
            raise ValueError("Numerical issue: negative diagonal entry in A^{-1}.")
        delta = sqrt_term * float(np.sqrt(max(di, 0.0)))
        center = -float(Ainv_b[i])
        l[i] = center - delta
        u[i] = center + delta

    return l, u


# =============================================================================
# PSD constructor that preserves sparsity for graph-like patterns
# =============================================================================

def _pattern_laplacian_psd_from_pattern(A_pattern: np.ndarray, rng: np.random.Generator, eps: float = 1e-8):
    """
    Build PSD matrix with same off-diagonal sparsity pattern as A_pattern using a weighted Laplacian.
    """
    A_pattern = np.asarray(A_pattern, dtype=float)
    n = A_pattern.shape[0]
    P = (A_pattern != 0).astype(float)
    np.fill_diagonal(P, 0.0)

    W = rng.uniform(0.5, 2.0, size=(n, n))
    W = np.where(P > 0, W, 0.0)
    W = np.tril(W) + np.triu(W.T, 1)

    d = W.sum(axis=1)
    L = np.diag(d) - W  # PSD
    if eps > 0:
        L = L + eps * np.eye(n)
    return L


# =============================================================================
# Data structure
# =============================================================================

@dataclass
class QCQPInstance:
    instance_id: int
    bounds_exist: bool
    A_list: list
    b_list: list
    c_list: list
    l: np.ndarray
    u: np.ndarray
    results: dict = field(default_factory=dict)

    n: Optional[int] = None
    m: Optional[int] = None
    group_id: Optional[int] = None
    size_tag: Optional[str] = None

    l_art: Optional[np.ndarray] = None
    u_art: Optional[np.ndarray] = None


# =============================================================================
# Generator (single size)
# =============================================================================

class QCQPInstanceGenerator:
    def __init__(
        self,
        n,
        m,
        num_instances,
        seed,
        outcome_weights,
        matrix_weights=None,
        p_linear_objective: float = 0.86,   # MINLPLib-like
        p_convex_constr: float = 0.50,      # convex prominent
        p_concave_constr: float = 0.10,     # concave small
    ):
        self.n = int(n)
        self.m = int(m)
        self.master_seed = int(seed)
        self.num_instances = int(num_instances)
        self.outcome_weights = np.asarray(outcome_weights, dtype=float)
        self.matrix_weights = matrix_weights

        self.p_linear_objective = float(p_linear_objective)
        self.p_convex_constr = float(p_convex_constr)
        self.p_concave_constr = float(p_concave_constr)

    def _base_seed(self):
        return int(self.master_seed) + 1_000_003 * int(self.m) + 100_069 * int(self.n)

    def determine_outcome(self, instance_id):
        local_seed = self._base_seed() + int(instance_id) * 13 + 7
        rng = np.random.default_rng(local_seed)
        outcome = rng.choice(
            a=np.arange(1, len(self.outcome_weights) + 1),
            p=self.outcome_weights / np.sum(self.outcome_weights),
        )
        return int(outcome)

    def generate_matrix(self, matrix_type, bilinear, rng, n_neg=None):
        MATRIX_FUNCS = {
            "bipartite": generate_bipartite_distance_matrix,
            "tree": generate_tree_matrix,
            "planar": generate_planar_distance_matrix,
            "chordal": generate_chordal_distance_matrix,
            "symmetric": generate_symmetric_matrix,
            "diag_ord1": diag_ordered_ones,
            "diag_rand1": diag_random_ones,
            "diag_randI": diag_random_ints,
            "eig_ones": eigvals_ones,
            "eig_ints": eigvals_ints,
            "spd": generate_spd_matrix,
        }
        SIGN_TUNABLE = {"diag_ord1", "diag_rand1", "diag_randI", "eig_ones", "eig_ints"}

        if matrix_type in SIGN_TUNABLE:
            A, _ = MATRIX_FUNCS[matrix_type](self.n, rng, n_neg=n_neg)
        else:
            A, _ = MATRIX_FUNCS[matrix_type](self.n, rng)

        if bilinear:
            random_values = rng.uniform(-2, 3, size=(self.n, self.n))
            A = np.where(A != 0, random_values, 0.0)
            A = 0.5 * (A + A.T)
            np.fill_diagonal(A, 0.0)

        return A

    def generate_psd_matrix(self, matrix_type, rng):
        """
        PSD version of matrix_type:
        - spd: SPD
        - diag/eig sign-tunable: force n_neg=0 (PSD)
        - graph-types: Laplacian PSD with same pattern
        - symmetric: M^T M PSD
        """
        n = self.n
        SIGN_TUNABLE = {"diag_ord1", "diag_rand1", "diag_randI", "eig_ones", "eig_ints"}
        GRAPH_TYPES = {"bipartite", "tree", "planar", "chordal"}

        if matrix_type == "spd":
            A, _ = generate_spd_matrix(n, rng)
            return A

        if matrix_type in SIGN_TUNABLE:
            if matrix_type == "diag_ord1":
                A, _ = diag_ordered_ones(n, rng, n_neg=0)
                return A
            if matrix_type == "diag_rand1":
                A, _ = diag_random_ones(n, rng, n_neg=0)
                return A
            if matrix_type == "diag_randI":
                A, _ = diag_random_ints(n, rng, n_neg=0)
                return A
            if matrix_type == "eig_ones":
                A, _ = eigvals_ones(n, rng, n_neg=0)
                return A
            if matrix_type == "eig_ints":
                A, _ = eigvals_ints(n, rng, n_neg=0)
                return A

        if matrix_type in GRAPH_TYPES:
            # generate a pattern matrix, then build PSD Laplacian with same pattern
            if matrix_type == "bipartite":
                A_pat, _ = generate_bipartite_distance_matrix(n, rng)
            elif matrix_type == "tree":
                A_pat, _ = generate_tree_matrix(n, rng)
            elif matrix_type == "planar":
                A_pat, _ = generate_planar_distance_matrix(n, rng)
            else:
                A_pat, _ = generate_chordal_distance_matrix(n, rng)
            return _pattern_laplacian_psd_from_pattern(A_pat, rng, eps=1e-8)

        # symmetric / fallback: PSD dense
        M = rng.uniform(-2.0, 2.0, size=(n, n))
        A = M.T @ M + 1e-8 * np.eye(n)
        return 0.5 * (A + A.T)

    def generate_instance(self, instance_id):
        base = self._base_seed()
        rng = np.random.default_rng(base + int(instance_id) * 10 + 3)

        outcome = self.determine_outcome(instance_id)

        bounds_exist = outcome > 4
        matrix_combined = outcome in {1, 2, 5, 6}
        bilinear = outcome in {1, 3, 5, 7}

        sparsity_matrices = ["bipartite", "tree", "planar", "chordal", "symmetric", "eig_ones", "eig_ints"]
        non_eligible_matrices = ["spd", "diag_ord1", "diag_rand1", "diag_randI"]
        all_matrices = sparsity_matrices + non_eligible_matrices

        A_list, b_list, c_list = [], [], []
        matrix_types = sparsity_matrices if bilinear else all_matrices

        if matrix_combined:
            weights = np.array([self.matrix_weights[mt] for mt in matrix_types], dtype=float)
            prob = weights / weights.sum()
            chosen_matrix_types = rng.choice(matrix_types, size=self.m, replace=True, p=prob)
        else:
            weights = np.array([self.matrix_weights[mt] for mt in all_matrices], dtype=float)
            prob = weights / weights.sum()
            single_type = rng.choice(all_matrices, p=prob)
            chosen_matrix_types = [single_type] * self.m

        SIGN_TUNABLE = {"diag_ord1", "diag_rand1", "diag_randI", "eig_ones", "eig_ints"}
        num_neg_eigvals = []
        for mt in chosen_matrix_types:
            n_neg = int(rng.integers(0, self.n + 1)) if mt in SIGN_TUNABLE else None
            num_neg_eigvals.append(n_neg)

        # bounds-missing: guarantee ONE PD CONSTRAINT at k>=1 and store l_art/u_art
        pd_index = None
        l_art = None
        u_art = None
        if not bounds_exist:
            pd_index = int(rng.integers(1, self.m)) if self.m > 1 else 0  # constraint-only when possible

        for i, matrix_type in enumerate(chosen_matrix_types):
            sub_rng = np.random.default_rng(base + int(instance_id) * 1000 + i * 97 + 11)
            n_neg = num_neg_eigvals[i]

            # (A) PD injection FIRST (cannot be broken by linear objective)
            if (not bounds_exist) and (i == pd_index):
                while True:
                    A_pd, _ = generate_spd_matrix(self.n, sub_rng)
                    b_pd = sub_rng.uniform(-5, 5, size=self.n)
                    c_pd = -sub_rng.uniform(0, 10)
                    try:
                        np.linalg.cholesky(A_pd)
                        l_new, u_new = compute_artificial_bounds(A_pd, b_pd, c_pd)
                        break
                    except Exception:
                        continue
                A_list.append(A_pd)
                b_list.append(b_pd)
                c_list.append(c_pd)
                l_art, u_art = l_new, u_new
                continue

            # (B) Objective: make linear with prob p_linear_objective
            if i == 0 and (sub_rng.random() < self.p_linear_objective):
                A_list.append(np.zeros((self.n, self.n), dtype=float))
                b_list.append(sub_rng.uniform(-5, 5, size=self.n))
                c_list.append(-sub_rng.uniform(0, 10))
                continue

            # (C) Constraints: convex/concave/indef control (only if NOT bilinear)
            if i >= 1 and (not bilinear):
                r = sub_rng.random()
                if r < self.p_convex_constr:
                    A = self.generate_psd_matrix(matrix_type, sub_rng)
                elif r < self.p_convex_constr + self.p_concave_constr:
                    A = -self.generate_psd_matrix(matrix_type, sub_rng)
                else:
                    A = self.generate_matrix(matrix_type, bilinear, sub_rng, n_neg=n_neg)
            else:
                # bilinear constraints stay as-is (indef), and objective stays as-is if not linear
                A = self.generate_matrix(matrix_type, bilinear, sub_rng, n_neg=n_neg)

            A_list.append(A)
            b_list.append(sub_rng.uniform(-5, 5, size=self.n))
            c_list.append(-sub_rng.uniform(0, 10))

        # ORIGINAL bounds only
        if bounds_exist:
            l = np.zeros(self.n, dtype=float)
            u = np.ones(self.n, dtype=float)
        else:
            l = np.full(self.n, -np.inf, dtype=float)
            u = np.full(self.n,  np.inf, dtype=float)

        inst = QCQPInstance(int(instance_id), bool(bounds_exist), A_list, b_list, c_list, l, u)
        inst.n = self.n
        inst.m = self.m

        if not bounds_exist:
            inst.l_art = l_art
            inst.u_art = u_art

        return inst

    def save_instances(self, filename, start_id=0):
        instances = [self.generate_instance(i) for i in range(int(start_id), int(start_id) + self.num_instances)]
        pd.to_pickle(instances, filename)


# =============================================================================
# Generator (mixed sizes)
# =============================================================================

class QCQPSizeMixtureGenerator:
    """
    size_plan entries like:
        {"n": 5, "m_constr": 10, "count": 400}
    Optional overrides:
        {"outcome_weights":..., "matrix_weights":..., "p_linear_objective":..., "p_convex_constr":..., "p_concave_constr":...}
    """

    def __init__(
        self,
        size_plan: List[Dict[str, Any]],
        seed: int,
        default_outcome_weights: np.ndarray,
        default_matrix_weights: Dict[str, float],
        default_p_linear_objective: float = 0.86,
        default_p_convex_constr: float = 0.50,
        default_p_concave_constr: float = 0.10,
        shuffle: bool = True,
        shuffle_seed_offset: int = 12345,
    ):
        self.master_seed = int(seed)
        self.size_plan = list(size_plan)
        self.default_outcome_weights = np.asarray(default_outcome_weights, dtype=float)
        self.default_matrix_weights = dict(default_matrix_weights)

        self.default_p_linear_objective = float(default_p_linear_objective)
        self.default_p_convex_constr = float(default_p_convex_constr)
        self.default_p_concave_constr = float(default_p_concave_constr)

        self._gens: Dict[int, QCQPInstanceGenerator] = {}
        jobs: List[Tuple[int, int]] = []

        for gid, spec in enumerate(self.size_plan):
            n = int(spec["n"])
            m_constr = int(spec["m_constr"])
            count = int(spec["count"])
            m_total = m_constr + 1

            ow = np.asarray(spec.get("outcome_weights", self.default_outcome_weights), dtype=float)
            mw = spec.get("matrix_weights", self.default_matrix_weights)

            plo = float(spec.get("p_linear_objective", self.default_p_linear_objective))
            pcv = float(spec.get("p_convex_constr", self.default_p_convex_constr))
            pcc = float(spec.get("p_concave_constr", self.default_p_concave_constr))

            gen = QCQPInstanceGenerator(
                n=n,
                m=m_total,
                num_instances=count,
                seed=self.master_seed,
                outcome_weights=ow,
                matrix_weights=mw,
                p_linear_objective=plo,
                p_convex_constr=pcv,
                p_concave_constr=pcc,
            )
            self._gens[gid] = gen

            for local_id in range(count):
                jobs.append((gid, local_id))

        if shuffle:
            rng = np.random.default_rng(self.master_seed + int(shuffle_seed_offset))
            perm = rng.permutation(len(jobs))
            self.jobs = [jobs[i] for i in perm]
        else:
            self.jobs = jobs

        self.num_instances = len(self.jobs)

    def generate_instance(self, global_id: int) -> QCQPInstance:
        gid, local_id = self.jobs[int(global_id)]
        inst = self._gens[gid].generate_instance(local_id)

        inst.instance_id = int(global_id)
        inst.group_id = int(gid)
        inst.n = inst.l.size
        inst.m = len(inst.A_list)
        inst.size_tag = f"n{inst.n}_m{inst.m}"
        return inst

    def save_instances(self, filename: str, start_id: int = 0):
        start_id = int(start_id)
        end_id = min(self.num_instances, start_id + self.num_instances)
        instances = [self.generate_instance(i) for i in range(start_id, end_id)]
        pd.to_pickle(instances, filename)

In [2]:
# =========================
# CELL 1 — BUILD + SAVE MIXED-SIZE PKL
# =========================
if __name__ == "__main__":
    import os, gc, psutil, pickle
    from qcqp_solversdg import QCQPSolver

    custom_matrix_weights = {
        "bipartite": 1,
        "tree": 1,
        "planar": 1,
        "chordal": 1,
        "symmetric": 1,
        "spd": 10,
        "diag_ord1": 1,
        "diag_rand1": 1,
        "diag_randI": 1,
        "eig_ones": 1,
        "eig_ints": 1,
    }

    outcome_weights = np.array([
        0.04705882, 0.02941176, 0.04117647, 0.02941176,  # outcomes 1-4 (no bounds)  sum=0.14705882
        0.22745098, 0.11372549, 0.28431373, 0.22745098   # outcomes 5-8 (bounds)     sum=0.85294118
    ], dtype=float)

    size_plan = [
        {'n': 10, 'm_constr': 18, 'count': 400},
        {'n': 9,  'm_constr': 10, 'count': 400},
        {'n': 16, 'm_constr': 48, 'count': 400},
        {'n': 43, 'm_constr': 84, 'count': 400},
        {'n': 14, 'm_constr': 28, 'count': 400},
        {'n': 33, 'm_constr': 71, 'count': 400},
        {'n': 18, 'm_constr': 56, 'count': 400},
        {'n': 4,  'm_constr': 2,  'count': 400},
        {'n': 14, 'm_constr': 36, 'count': 400},
        {'n': 14, 'm_constr': 26, 'count': 400},
        {'n': 14, 'm_constr': 22, 'count': 400},
        {'n': 12, 'm_constr': 16, 'count': 400},
        {'n': 17, 'm_constr': 36, 'count': 400},
        {'n': 12, 'm_constr': 26, 'count': 400},
        {'n': 14, 'm_constr': 23, 'count': 400},
        {'n': 19, 'm_constr': 25, 'count': 400},
        {'n': 33, 'm_constr': 79, 'count': 400},
        {'n': 10, 'm_constr': 12, 'count': 400},
        {'n': 5,  'm_constr': 3,  'count': 400},
        {'n': 18, 'm_constr': 37, 'count': 400},
    ]

    gen = QCQPSizeMixtureGenerator(
        size_plan=size_plan,
        seed=42,
        default_outcome_weights=outcome_weights,
        default_matrix_weights=custom_matrix_weights,
        shuffle=True,
    )

    pkl_path = "instances_mixed7.pkl"
    gen.save_instances(pkl_path)
    print("✅ saved:", pkl_path, " | num_instances =", gen.num_instances)


✅ saved: instances_mixed7.pkl  | num_instances = 8000


In [3]:

    # =========================
    # CELL 2 — LOAD PKL + SOLVE + WRITE CSV
    # =========================

    def _has_finite_bounds(l, u):
        l = np.asarray(l, dtype=float).reshape(-1)
        u = np.asarray(u, dtype=float).reshape(-1)
        return (l.size > 0 and u.size == l.size and np.all(np.isfinite(l)) and np.all(np.isfinite(u)))

    def _bounds_valid(l, u):
        l = np.asarray(l, dtype=float).reshape(-1)
        u = np.asarray(u, dtype=float).reshape(-1)
        if l.size == 0 or u.size == 0 or l.size != u.size:
            return False
        if not (np.all(np.isfinite(l)) and np.all(np.isfinite(u))):
            return False
        if not np.all(u > l):
            return False
        return True

    def _working_bounds_for_solve(inst_d):
        # Use original bounds if present
        l = np.asarray(inst_d.get("l", []), dtype=float).reshape(-1)
        u = np.asarray(inst_d.get("u", []), dtype=float).reshape(-1)
        bounds_exist = bool(inst_d.get("bounds_exist", False))

        if bounds_exist and _has_finite_bounds(l, u):
            return l, u

        # Use cached artificial bounds (computed ONCE in generator)
        l_art = inst_d.get("l_art", None)
        u_art = inst_d.get("u_art", None)
        if l_art is not None and u_art is not None and _bounds_valid(l_art, u_art):
            return np.asarray(l_art, dtype=float).reshape(-1), np.asarray(u_art, dtype=float).reshape(-1)

        raise RuntimeError("Missing l_art/u_art for bounds_exist=False instance (should not happen with redesigned generator).")

    def run_instances_from_pkl(
        pkl_path,
        out_csv="instancesobj_mixed_train7.csv",
        solve_lp=True,
        solve_sdp=True,
        solve_sdp_prime=True,
        solve_sdp_rlt=True,
        print_every=10,
        gc_every=200,
        solver_kwargs=None,
    ):
        if solver_kwargs is None:
            solver_kwargs = {}

        with open(pkl_path, "rb") as f:
            instances = pickle.load(f)

        if out_csv and os.path.exists(out_csv):
            os.remove(out_csv)

        header_written = False
        proc = psutil.Process(os.getpid())

        split = ""
        if out_csv:
            if "tst" in out_csv:
                split = "test"
            elif "tr" in out_csv or "train" in out_csv:
                split = "train"

        for idx, inst in enumerate(instances):
            try:
                inst_d = inst.__dict__ if hasattr(inst, "__dict__") else inst

                A_list = inst_d["A_list"]
                b_list = inst_d["b_list"]
                c_list = inst_d["c_list"]

                l_work, u_work = _working_bounds_for_solve(inst_d)

                n = int(inst_d.get("n", None) or l_work.size or len(b_list[0]))
                m_total = int(inst_d.get("m", None) or len(A_list))
                m_constr = m_total - 1

                solver = QCQPSolver(A_list, b_list, c_list, l_work, u_work, **solver_kwargs)

                lp  = solver.solve("LP")  if solve_lp  else {"objective": np.nan, "time": np.nan, "status":"skipped"}
                sdp = solver.solve("SDP") if solve_sdp else {"objective": np.nan, "time": np.nan, "status":"skipped"}

                sdp_p = solver.solve("SDP_PRIME") if solve_sdp_prime else {"objective": np.nan, "time": np.nan, "status": "skipped"}
                sdp_rlt = solver.solve("SDP_RLT") if solve_sdp_rlt else {"objective": np.nan, "time": np.nan, "status": "skipped"}

                if print_every and (idx % print_every == 0):
                    print(
                        f"idx={idx} id={inst_d.get('instance_id', idx)} | n={n} m_constr={m_constr} | "
                        f"LP={lp.get('objective')} | SDP={sdp.get('objective')}",
                        flush=True,
                    )

                row = {
                    "id": int(inst_d.get("instance_id", idx)),
                    "split": split,
                    "seed": inst_d.get("seed", None),
                    "id_global": inst_d.get("id_global", f"{inst_d.get('instance_id', idx)}_n{n}_mc{m_constr}"),
                    "group_id": inst_d.get("group_id", None),
                    "size_tag": inst_d.get("size_tag", None),
                    "bounds_exist": inst_d.get("bounds_exist", None),
                    "n": n,
                    "m_total": m_total,
                    "m_constr": m_constr,
                    "lp_obj": lp.get("objective"),
                    "lp_t": lp.get("time"),
                    "lp_status": lp.get("status", "ok"),
                    "sdp_obj": sdp.get("objective"),
                    "sdp_t": sdp.get("time"),
                    "sdp_status": sdp.get("status", "ok"),
                    "sdp_prime_obj": sdp_p.get("objective"),
                    "sdp_prime_t": sdp_p.get("time"),
                    "sdp_prime_status": sdp_p.get("status", "ok"),
                    "sdp_rlt_obj": sdp_rlt.get("objective"),
                    "sdp_rlt_t": sdp_rlt.get("time"),
                    "sdp_rlt_status": sdp_rlt.get("status", "ok"),
                }

            except Exception as e:
                print(f"⚠️ idx={idx} failed: {e}", flush=True)
                row = {
                    "id": int(getattr(inst, "instance_id", idx) if hasattr(inst, "instance_id") else idx),
                    "split": split,
                    "seed": None,
                    "id_global": f"err_{idx}",
                    "group_id": None,
                    "size_tag": None,
                    "bounds_exist": None,
                    "n": None,
                    "m_total": None,
                    "m_constr": None,
                    "lp_obj": np.nan,
                    "lp_t": np.nan,
                    "lp_status": f"error:{e}",
                    "sdp_obj": np.nan,
                    "sdp_t": np.nan,
                    "sdp_status": f"error:{e}",
                    "sdp_prime_obj": np.nan,
                    "sdp_prime_t": np.nan,
                    "sdp_prime_status": f"error:{e}",
                    "sdp_rlt_obj": np.nan,
                    "sdp_rlt_t": np.nan,
                    "sdp_rlt_status": f"error:{e}",
                }

            if out_csv:
                pd.DataFrame([row]).to_csv(out_csv, mode="a", index=False, header=not header_written)
                header_written = True

            try:
                del solver
            except Exception:
                pass

            if gc_every and ((idx + 1) % gc_every == 0):
                gc.collect()

            if (idx + 1) % 500 == 0:
                print("MEM_RSS_MB=", proc.memory_info().rss / 1e6, flush=True)

        print("✅ wrote:", out_csv)

    run_instances_from_pkl(
        pkl_path=pkl_path,
        out_csv="instancesobj_mixed_train7.csv",
        print_every=50,
        solver_kwargs={
            "use_blocks": True,
            "block_tol": 1e-8,
            "verbose_blocks": False,
            "prune_matrices": False,
            "min_block_edges": 0,
        },
        solve_lp=True,
        solve_sdp=True,
        solve_sdp_prime=True,
        solve_sdp_rlt=True,
    )

Set parameter Username
Academic license - for non-commercial use only - expires 2026-09-17
idx=0 id=0 | n=16 m_constr=48 | LP=-9.09021615108517 | SDP=-8.91920255946338
idx=50 id=50 | n=12 m_constr=26 | LP=-34.904628685574565 | SDP=-34.90462866892478
idx=100 id=100 | n=19 m_constr=25 | LP=-85.20527388385139 | SDP=-30.602576481044558
idx=150 id=150 | n=4 m_constr=2 | LP=-13.595279266016389 | SDP=-13.59527926601639
idx=200 id=200 | n=43 m_constr=84 | LP=-892.6895241205539 | SDP=-35.67900575305438
idx=250 id=250 | n=10 m_constr=12 | LP=-17.684460010964763 | SDP=-13.017839741447256
idx=300 id=300 | n=14 m_constr=22 | LP=-1388.73017553561 | SDP=-285.56561555618924
idx=350 id=350 | n=14 m_constr=28 | LP=-11.00001547124679 | SDP=-45.89938106384902
idx=400 id=400 | n=14 m_constr=22 | LP=-20.015759161363068 | SDP=-11.106805679545685
idx=450 id=450 | n=5 m_constr=3 | LP=-28.010919050820167 | SDP=-28.01091905061984
MEM_RSS_MB= 2725.19168
idx=500 id=500 | n=14 m_constr=36 | LP=-5.606409027613022 | 

In [4]:
"""import pandas as pd

# Load the saved instances
instances = pd.read_pickle("instances_5_100tr.pkl")

# Choose the instance you want to inspect
index = 0  # Change this to any index you want
inst = instances[index]

print(f"📌 Instance ID: {inst.instance_id}")
print(f"Bounds Exist?: {inst.bounds_exist}\n")"""

'import pandas as pd\n\n# Load the saved instances\ninstances = pd.read_pickle("instances_5_100tr.pkl")\n\n# Choose the instance you want to inspect\nindex = 0  # Change this to any index you want\ninst = instances[index]\n\nprint(f"📌 Instance ID: {inst.instance_id}")\nprint(f"Bounds Exist?: {inst.bounds_exist}\n")'